# Budget-constrained R&D portfolio decision

This notebook asks which candidates should advance under finite downstream capacity and which follow-up experiment is worth buying before that decision. The case is a **controlled synthetic portfolio experiment** because public Crop Protection datasets do not expose candidate-level proprietary development economics and matched efficacy/safety packages.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from crop_protection_ps.portfolio_decision import PortfolioConfig
from crop_protection_ps.portfolio_demo import run_portfolio_demo

config = PortfolioConfig()
{
    "candidates": config.n_candidates,
    "max_advanced": config.max_advanced,
    "followup_budget": config.followup_budget_units,
    "downstream_budget": config.downstream_budget_units,
    "rollouts": config.n_rollouts,
}

{'candidates': 36,
 'max_advanced': 6,
 'followup_budget': 24,
 'downstream_budget': 130,
 'rollouts': 500}

## Decision model

For candidate $i$, technical success requires both latent efficacy and safety-margin thresholds to be exceeded. Posterior expected development value is

\[u_i=P(T_i=1\mid D)V_i-C_i.\]

The terminal advancement decision is an exact 0/1 knapsack with both a development-budget constraint and a maximum number of programmes.

In [2]:
summary = run_portfolio_demo(ROOT)
summary["decision_structure"]

{'technical_success': 'efficacy > threshold AND safety margin > threshold',
 'candidate_expected_value': 'P(technical success) * success reward - development cost',
 'terminal_optimisation': 'exact count-and-budget 0/1 knapsack',
 'adaptive_acquisition': 'Gauss-Hermite EVSI approximation around the current portfolio value-per-cost boundary'}

## Equal follow-up budget

Uniform, uncertainty-only and portfolio-VOI policies each spend exactly 24 follow-up cost units. The zero-follow-up policy is shown only as a reference and is not an equal-budget comparator.

In [3]:
metrics = pd.read_csv(ROOT / "results" / "portfolio_decision" / "portfolio_policy_metrics.csv")
metrics[[
    "policy",
    "mean_followup_cost_units",
    "mean_n_efficacy_followups",
    "mean_n_safety_followups",
    "mean_realised_portfolio_value",
    "mean_oracle_regret",
    "mean_advanced_technical_success_rate",
]]

,policy,mean_followup_cost_units,mean_n_efficacy_followups,mean_n_safety_followups,mean_realised_portfolio_value,mean_oracle_regret,mean_advanced_technical_success_rate
0,no_followup,0.0,0.000,0.000,232.818095,161.894265,0.726433
1,portfolio_voi,24.0,2.012,7.976,289.267166,105.445194,0.833533
2,uncertainty,24.0,0.522,10.956,268.061371,126.650989,0.809367
3,uniform,24.0,4.000,4.000,245.205732,149.506628,0.750733


## Why uncertainty and value of information differ

The uncertainty policy asks which affordable experiment most reduces entropy in technical success. Portfolio-VOI asks whether the posterior change could alter a candidate's position relative to the current constrained portfolio boundary. It therefore includes reward, development cost and scarce portfolio capacity.

In [4]:
summary["promotion_gate"]

{'minimum_regret_reduction_vs_uncertainty_percent': 5.0,
 'observed_regret_reduction_vs_uncertainty_percent': 16.743489290409656,
 'observed_regret_reduction_vs_uniform_percent': 29.471224637502715,
 'paired_regret_difference_portfolio_voi_minus_uncertainty': {'mean_difference': -21.205794717521293,
  'mc_standard_error': 2.6282733634217896,
  'mc95_low': -26.357210509828,
  'mc95_high': -16.054378925214586},
 'paired_regret_difference_portfolio_voi_minus_uniform': {'mean_difference': -44.06143422706524,
  'mc_standard_error': 3.2206459597822836,
  'mc95_low': -50.37390030823852,
  'mc95_high': -37.748968145891965},
 'promoted': True}

The promotion rule is tied to the **final decision**: portfolio-VOI must reduce oracle regret by at least 5% versus uncertainty sampling, and paired 95% Monte Carlo intervals must remain below zero versus both equal-budget comparators.

In [5]:
summary["equal_followup_budget_result"]

{'uniform': {'mean_followup_cost_units': 24.0,
  'mean_realised_portfolio_value': 245.20573160600185,
  'mean_oracle_regret': 149.50662813989805,
  'mean_advanced_technical_success_rate': 0.7507333333333334,
  'mean_efficacy_followups': 4.0,
  'mean_safety_followups': 4.0},
 'uncertainty': {'mean_followup_cost_units': 24.0,
  'mean_realised_portfolio_value': 268.06137111554574,
  'mean_oracle_regret': 126.65098863035415,
  'mean_advanced_technical_success_rate': 0.8093666666666667,
  'mean_efficacy_followups': 0.522,
  'mean_safety_followups': 10.956},
 'portfolio_voi': {'mean_followup_cost_units': 24.0,
  'mean_realised_portfolio_value': 289.2671658330671,
  'mean_oracle_regret': 105.44519391283285,
  'mean_advanced_technical_success_rate': 0.8335333333333333,
  'mean_efficacy_followups': 2.012,
  'mean_safety_followups': 7.976}}

## Interpretation

The controlled result demonstrates that reducing scientific uncertainty is not necessarily the same as reducing decision uncertainty. The terminal optimiser is exact; the experiment-acquisition rule is explicitly an approximation evaluated with deterministic Gauss-Hermite quadrature.

The rewards, costs, thresholds and numerical gains are simulation parameters, **not** estimates of real R&D economics or claims about a real active-ingredient portfolio.